In [ ]:
from kan import *
from kan.spline import B_batch, coef2curve, curve2coef, extend_grid
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cpu


# Create the Dataset
- Generates random input points in [-1, 1], evaluates f on them, and splits into train/test

In [ ]:
f = lambda x: torch.exp(torch.sin(torch.pi*x[:,[0]]) + x[:,[1]]**2)
dataset = create_dataset(f, n_var=2, train_num=1000, device=device)

print('train input:', dataset['train_input'].shape)
print('train label:', dataset['train_label'].shape)


train input: torch.Size([1000, 2])
train label: torch.Size([1000, 1])


# B-Spline Activation Function

In [ ]:
class BSplineActivation(nn.Module):
    def __init__(self, in_features, grid=3, k=3):
        super().__init__()
        self.in_features = in_features
        self.grid_size = grid
        self.k = k

        h = 2.0 / grid
        grid_pts = torch.linspace(-1 - k*h, 1 + k*h, grid + 2*k + 1)
        self.register_buffer('grid', grid_pts.unsqueeze(0).expand(in_features, -1).clone())
        self.coef = nn.Parameter(torch.zeros(in_features, grid + k))

    def forward(self, x):
        basis = B_batch(x, self.grid, k=self.k)
        return torch.einsum('bik,ik->bi', basis, self.coef)

    def update_grid_from_samples(self, x):
        with torch.no_grad():
            num = self.grid_size
            k = self.k

            x_sorted, _ = torch.sort(x, dim=0)
            ids = torch.linspace(0, x.shape[0] - 1, num + 1).long()
            grid_adaptive = x_sorted[ids, :].T          # (in_features, num+1)

            margin = 0.01
            h = (grid_adaptive[:, [-1]] - grid_adaptive[:, [0]] + 2 * margin) / num
            grid_uniform = grid_adaptive[:, [0]] - margin + h * torch.arange(num + 1).to(x.device)

            grid_eps = 0.02
            grid_base = grid_eps * grid_uniform + (1 - grid_eps) * grid_adaptive
            grid_extended = extend_grid(grid_base, k_extend=k)
            self.register_buffer('grid', grid_extended.to(x.device))


In [ ]:
class BSplineMLP(nn.Module):
    def __init__(self, width, grid=3, k=3, seed=0, device='cpu'):
        super().__init__()
        torch.manual_seed(seed)
        self.width = width
        self.depth = len(width) - 1
        self.k = k
        self.device = device
        self.cache_data = None

        linears = []
        activations = []
        for i in range(self.depth - 1):
            linears.append(nn.Linear(width[i], width[i+1]))
            activations.append(BSplineActivation(width[i+1], grid=grid, k=k))
        linears.append(nn.Linear(width[-2], width[-1]))

        self.linears = nn.ModuleList(linears)
        self.activations = nn.ModuleList(activations)
        self.to(device)

    def forward(self, x):
        for i in range(self.depth - 1):
            x = self.linears[i](x)
            x = self.activations[i](x)
        x = self.linears[-1](x)
        return x

    def update_grids(self, x):
        with torch.no_grad():
            for i in range(self.depth - 1):
                x = self.linears[i](x)
                self.activations[i].update_grid_from_samples(x)
                x = self.activations[i](x)

    def refine(self, new_grid):
        with torch.no_grad():
            x = self.cache_data
            for i in range(self.depth - 1):
                x = self.linears[i](x)
                pre_act = x.clone()

                act = self.activations[i]
                k = act.k
                num = new_grid

                x_sorted, _ = torch.sort(pre_act, dim=0)
                ids = torch.linspace(0, pre_act.shape[0] - 1, num + 1).long()
                grid_adaptive = x_sorted[ids, :].T

                margin = 0.01
                h = (grid_adaptive[:, [-1]] - grid_adaptive[:, [0]] + 2 * margin) / num
                grid_uniform = grid_adaptive[:, [0]] - margin + h * torch.arange(num + 1).to(self.device)

                grid_eps = 0.02
                grid_base = grid_eps * grid_uniform + (1 - grid_eps) * grid_adaptive
                new_grid_buf = extend_grid(grid_base, k_extend=k).to(self.device)

                y_current = act(pre_act)
                y_3d = y_current.unsqueeze(2)
                new_coef = curve2coef(pre_act, y_3d, new_grid_buf, k)

                act.register_buffer('grid', new_grid_buf)
                act.coef = nn.Parameter(new_coef[:, 0, :])
                act.grid_size = new_grid

                x = act(pre_act)
        return self

    def fit(self, dataset, opt="LBFGS", steps=100, lr=1.):
        self.cache_data = dataset['train_input'].to(self.device)
        self.update_grids(self.cache_data)

        optimizer = torch.optim.LBFGS(self.parameters(), lr=lr,
                                       max_iter=20,
                                       line_search_fn="strong_wolfe")
        loss_fn = lambda pred, y: torch.mean((pred - y) ** 2)
        results = {'train_loss': [], 'test_loss': []}
        pbar = tqdm(range(steps), desc='description', ncols=100)

        train_input = dataset['train_input'].to(self.device)
        train_label = dataset['train_label'].to(self.device)
        test_input  = dataset['test_input'].to(self.device)
        test_label  = dataset['test_label'].to(self.device)

        def closure():
            optimizer.zero_grad()
            loss = loss_fn(self.forward(train_input), train_label)
            loss.backward()
            return loss

        for _ in pbar:
            optimizer.step(closure)
            with torch.no_grad():
                train_loss = torch.sqrt(loss_fn(self.forward(train_input), train_label))
                test_loss  = torch.sqrt(loss_fn(self.forward(test_input),  test_label))
            results['train_loss'].append(train_loss.item())
            results['test_loss'].append(test_loss.item())
            pbar.set_description("| train_loss: %.2e | test_loss: %.2e |" % (train_loss, test_loss))

        return results


In [ ]:
model_mlp = BSplineMLP(width=[2, 20, 1], grid=3, k=3, seed=1, device=device)

print("--- Training at grid=3 ---")
model_mlp.fit(dataset, opt="LBFGS", steps=20)

print("\n--- Refining to grid=10 ---")
model_mlp = model_mlp.refine(10)

print("\n--- Training at grid=10 ---")
model_mlp.fit(dataset, opt="LBFGS", steps=20)


--- Training at grid=3 ---


| train_loss: 2.64e-02 | test_loss: 2.84e-02 |: 100%|███████████████| 20/20 [00:01<00:00, 15.72it/s]



--- Refining to grid=10 ---

--- Training at grid=10 ---


| train_loss: 7.18e-03 | test_loss: 8.47e-03 |: 100%|███████████████| 20/20 [00:01<00:00, 13.32it/s]


{'train_loss': [0.024016641080379486,
  0.020833417773246765,
  0.01761423982679844,
  0.014915370382368565,
  0.012928362004458904,
  0.012089813128113747,
  0.011455506086349487,
  0.010688645765185356,
  0.010035128332674503,
  0.009517916478216648,
  0.009208899922668934,
  0.008839156478643417,
  0.00839744322001934,
  0.008089007809758186,
  0.007906980812549591,
  0.0077682603150606155,
  0.007707791402935982,
  0.007548402063548565,
  0.007350608240813017,
  0.007177472580224276],
 'test_loss': [0.02567102760076523,
  0.022364314645528793,
  0.01972053200006485,
  0.01630972884595394,
  0.013883176259696484,
  0.01308300718665123,
  0.012279864400625229,
  0.011631879024207592,
  0.010919501073658466,
  0.01034460123628378,
  0.010088645853102207,
  0.009743648581206799,
  0.009586852043867111,
  0.00933580007404089,
  0.009136326611042023,
  0.009092515334486961,
  0.008980418555438519,
  0.008828134275972843,
  0.00848933681845665,
  0.008465602062642574]}

In [ ]:
grids = np.array([3, 5, 10, 20, 50, 100])

mlp_train_losses = []
mlp_test_losses  = []
steps = 200
k = 3

for i in range(len(grids)):
    if i == 0:
        model_mlp = BSplineMLP(width=[2, 20, 1], grid=grids[i], k=k, seed=0, device=device)
    else:
        model_mlp = model_mlp.refine(grids[i])
    results = model_mlp.fit(dataset, opt="LBFGS", steps=steps)
    mlp_train_losses += results['train_loss']
    mlp_test_losses  += results['test_loss']


| train_loss: 1.21e-02 | test_loss: 1.33e-02 |: 100%|█████████████| 200/200 [00:08<00:00, 23.75it/s]
| train_loss: 6.47e-03 | test_loss: 7.87e-03 |: 100%|█████████████| 200/200 [00:05<00:00, 35.15it/s]
| train_loss: 3.41e-03 | test_loss: 3.96e-03 |: 100%|█████████████| 200/200 [00:08<00:00, 24.49it/s]
| train_loss: 1.35e-03 | test_loss: 2.37e-03 |: 100%|█████████████| 200/200 [00:03<00:00, 52.46it/s]
| train_loss: 1.35e-03 | test_loss: 7.22e-03 |: 100%|█████████████| 200/200 [00:02<00:00, 76.06it/s]
| train_loss: 1.35e-03 | test_loss: 4.65e-02 |: 100%|█████████████| 200/200 [00:05<00:00, 38.83it/s]
